In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
%matplotlib inline


In [ ]:
# --- HYPERPARAMS ---
# Loan parameters
LOAN_AMOUNT = 10**6 * 10       # DKK (or your currency)
LOAN_INTEREST_RATE = 0.055     # % per year
LOAN_YEARS = 30                
LOAN_PRICE = 0.95              # Proportion of loan amount actually received (e.g., 0.95 means 5% lost instantly)

# Tax parameters
TAX_RATE = 0.22                # 22% on unrealised gains; interest is deductible

# Simulation parameters
SIMULATION_YEARS = 30          # Years to simulate and visualize
N_PATHS = 500                  # Number of Monte Carlo paths
START_DATE_MIN = 1920          # Minimum starting year to sample trajectories from

# Ongoing expenditure
YEARLY_EXPENDITURE = 0         # Ongoing yearly expense (e.g. living costs paid from portfolio)

assumed_inflation_rate = 0.02 #add this to the 
# Data path
CSV_PATH = Path("histretSP.xls") 
"""
url = "https://pages.stern.nyu.edu/~adamodar/pc/datasets/histretSP.xls"
"""

In [27]:

# Load without skipping to find the header index
df_raw = pd.read_excel("histretSP.xls", sheet_name="Returns by year", header=None, engine='xlrd')

# 1. Find the row index where the first column is "Year"
# We cast to string and strip whitespace to avoid mismatches
header_idx = df_raw[df_raw.iloc[:, 0].astype(str).str.strip() == 'Year'].index[0]

# 2. Re-load or re-assign columns based on that index
df = df_raw.iloc[header_idx:].copy()
df.columns = df.iloc[0] # Set the first row of this slice as headers
df = df.iloc[1:].reset_index(drop=True) # Drop the header row from the data

# 3. Clean and Cast
# Sometimes 'Year' column has trailing spaces in the header itself
df.columns = [str(c).strip() for c in df.columns]

df = df.dropna(subset=['Year'])
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df.dropna(subset=['Year']).reset_index(drop=True)

# 4. Fisher Equation for Real Return
# r_real = (1 + r_nom) / (1 + pi) - 1
df['Real Return'] = ((1 + df['S&P 500 (includes dividends)']) / (1 + df['Inflation Rate'])) - 1

# Resulting Matrix in R^{N x 4}
result_df = df[['Year', 'S&P 500 (includes dividends)', 'Inflation Rate', 'Real Return']]

print(f"Dataset Dimensions: {result_df.shape}")
print(result_df.tail(10))


result_df["Adjusted real return"] = result_df["Real Return"] + assumed_inflation_rate 

"""
I wanted to correct for historically periods of ultra high inflation rate.
So i take the inflation adjusted results and add a constant inflation rate of 2%.
This Adjusted real return is the number we should use in the calculation. 
"""

Dataset Dimensions: (98, 4)
      Year S&P 500 (includes dividends) Inflation Rate Real Return
88  2016.0                     0.117731       0.020746    0.095013
89  2017.0                     0.216055       0.021091    0.190937
90  2018.0                    -0.042269       0.019102    -0.06022
91  2019.0                     0.312117       0.022851    0.282803
92  2020.0                     0.180232        0.01362    0.164373
93  2021.0                     0.284689       0.070364    0.200235
94  2022.0                    -0.180375       0.064544   -0.230069
95  2023.0                     0.260607       0.033521     0.21972
96  2024.0                     0.248786       0.028881    0.213733
97  2025.0                     0.177237       0.027351    0.145895


'\nI wanted to correct for historically periods of ultra high inflation rate.\nSo i take the inflation adjusted results and add a constant inflation rate of 2%.\nThis Adjusted real return is the number we should use in the calculation. \n'

In [28]:
# We now use result_df which contains "Adjusted real return"
returns_decimal = result_df["Adjusted real return"].values

print(f"Loaded {len(returns_decimal)} years of S&P 500 returns (Adjusted real return).")
# valid_starts holds the indices of the dataframe corresponding to valid starting years
valid_starts = result_df[result_df["Year"] >= START_DATE_MIN].index.tolist()
print(f"{len(valid_starts)} valid starting years from {START_DATE_MIN} onwards.")


Loaded 98 years of S&P 500 returns (Adjusted real return).
98 valid starting years from 1920 onwards.


In [29]:
def loan_schedule(principal: float, annual_rate: float, years: int):
    """
    Annuity loan: same total payment each year. Returns arrays for each year (0..years-1):
    payment, interest, principal_paid, remaining_balance.
    """
    r = annual_rate
    n = years
    # Yearly payment (annuity): P * r * (1+r)^n / ((1+r)^n - 1)
    payment_per_year = principal * r * (1 + r) ** n / ((1 + r) ** n - 1)
    
    balance = principal
    payments = []
    interests = []
    principals = []
    balances = []
    
    for _ in range(n):
        interest = balance * r
        principal_paid = payment_per_year - interest
        balance = max(0, balance - principal_paid)
        payments.append(payment_per_year)
        interests.append(interest)
        principals.append(principal_paid)
        balances.append(balance)
    
    return {
        "payment": np.array(payments),
        "interest": np.array(interests),
        "principal": np.array(principals),
        "remaining_balance": np.array(balances),
    }

# Schedule is based on FULL LOAN_AMOUNT, even if we only receive LOAN_AMOUNT * LOAN_PRICE
schedule = loan_schedule(LOAN_AMOUNT, LOAN_INTEREST_RATE, LOAN_YEARS)
yearly_payment = schedule["payment"][0]
print(f"Assumptions: LOAN_AMOUNT={LOAN_AMOUNT:,.0f}, rate={LOAN_INTEREST_RATE:.2%}, term={LOAN_YEARS} yr.")
print(f"Yearly payment (same each year): {yearly_payment:,.0f}")
print(f"First year: interest {schedule['interest'][0]:,.0f}, principal {schedule['principal'][0]:,.0f}")
print(f"Last year: interest {schedule['interest'][-1]:,.0f}, principal {schedule['principal'][-1]:,.0f}")


Assumptions: LOAN_AMOUNT=10,000,000, rate=5.50%, term=30 yr.
Yearly payment (same each year): 688,054
First year: interest 550,000, principal 138,054
Last year: interest 35,870, principal 652,184


In [ ]:
def run_one_path(
    n_years: int,
    initial_portfolio: float,
    initial_loan_amount: float,
    yearly_payment_arr: np.ndarray,
    interest_arr: np.ndarray,
    balance_arr: np.ndarray,
    returns_sample: np.ndarray,
    tax_rate: float,
    yearly_expenditure: float,
):
    """
    Run one simulation path. returns_sample is length n_years (annual returns as decimals).

    Tax follows the Danish *lagerprincippet* for a company (ApS):
    taxable income each year = change in portfolio value over the year, minus deductible
    interest, minus any loss carried forward. Because sales proceeds are included in the
    value change, money withdrawn to pay the mortgage does NOT reduce the taxable gain.
    A negative result is not refunded: it becomes an underskud carried forward
    indefinitely and offsets future taxable income.
    """
    portfolio = initial_portfolio  # Start: we invest the received loan amount
    loss_carryforward = 0.0        # Unused interest deduction + losses (underskud til fremf\u00f8rsel)

    history = {
        "year": [0],
        "portfolio": [portfolio],
        "amount_owed": [initial_loan_amount],
        "net_gain": [portfolio - initial_loan_amount],
        "tax_paid_per_year": [0.0],
        "cumulative_tax": [0.0],
        "cumulative_interest_paid": [0.0],
        "loss_carryforward": [0.0],
    }
    cum_tax = 0.0
    cum_interest = 0.0

    for t in range(n_years):
        ret = returns_sample[t]
        portfolio_after_return = portfolio * (1 + ret)

        interest_paid = interest_arr[t] if t < len(interest_arr) else 0.0
        payment = yearly_payment_arr[t] if t < len(yearly_payment_arr) else 0.0
        balance = balance_arr[t] if t < len(balance_arr) else 0.0

        # --- Tax: lagerprincippet on the year's actual change in value ---------------
        # The gain is measured BEFORE this year's withdrawal, against the value the
        # portfolio actually started the year with. Withdrawals do not shelter gains.
        value_change = portfolio_after_return - portfolio
        taxable_income = value_change - interest_paid - loss_carryforward

        if taxable_income > 0.0:
            net_tax = tax_rate * taxable_income
            loss_carryforward = 0.0
        else:
            # No refund. The unused deduction (incl. unused interest) is carried forward.
            net_tax = 0.0
            loss_carryforward = -taxable_income

        cum_tax += net_tax
        cum_interest += interest_paid

        # Mortgage payment and any ongoing expenditure are funded by selling from the
        # portfolio. Under lagerprincippet selling triggers no extra tax event.
        portfolio = portfolio_after_return - net_tax - payment - yearly_expenditure
        portfolio = max(0.0, portfolio)

        net_gain = portfolio - balance
        history["year"].append(t + 1)
        history["portfolio"].append(portfolio)
        history["amount_owed"].append(balance)
        history["net_gain"].append(net_gain)
        history["cumulative_tax"].append(cum_tax)
        history["cumulative_interest_paid"].append(cum_interest)
        history["tax_paid_per_year"].append(net_tax)
        history["loss_carryforward"].append(loss_carryforward)

    return history

# Build loan arrays (same for all paths)
yearly_payment_arr = schedule["payment"]
interest_arr = schedule["interest"]
# balance at end of year t (after payment)
balance_arr = schedule["remaining_balance"]

# Extend loan arrays if we simulate more than LOAN_YEARS
if SIMULATION_YEARS > LOAN_YEARS:
    yearly_payment_arr = np.pad(yearly_payment_arr, (0, SIMULATION_YEARS - LOAN_YEARS), constant_values=0)
    interest_arr = np.pad(interest_arr, (0, SIMULATION_YEARS - LOAN_YEARS), constant_values=0)
    balance_arr = np.pad(balance_arr, (0, SIMULATION_YEARS - LOAN_YEARS), constant_values=0)
else:
    yearly_payment_arr = yearly_payment_arr[:SIMULATION_YEARS]
    interest_arr = interest_arr[:SIMULATION_YEARS]
    balance_arr = balance_arr[:SIMULATION_YEARS]


In [31]:
# Monte Carlo: sample starting year, follow trajectory.
# If trajectory goes out of bounds, resample starting year.
np.random.seed(42)
paths = []
initial_investment = LOAN_AMOUNT * LOAN_PRICE # Factoring in LOAN_PRICE (e.g. 5% lost instantly)

for _ in range(N_PATHS):
    returns_path = []
    
    current_idx = np.random.choice(valid_starts)
    while len(returns_path) < SIMULATION_YEARS:
        # Check if we hit the end of the dataset
        if current_idx >= len(returns_decimal):
            # Out of bounds, resample
            current_idx = np.random.choice(valid_starts)
            
        returns_path.append(returns_decimal[current_idx])
        current_idx += 1
        
    returns_path = np.array(returns_path)
    
    hist = run_one_path(
        SIMULATION_YEARS,
        initial_investment,
        LOAN_AMOUNT,
        yearly_payment_arr,
        interest_arr,
        balance_arr,
        returns_path,
        TAX_RATE,
        YEARLY_EXPENDITURE,
    )
    paths.append(hist)

# Aggregate for percentiles
portfolio_arr = np.array([p["portfolio"] for p in paths])
owed_arr = np.array([p["amount_owed"] for p in paths])
net_gain_arr = np.array([p["net_gain"] for p in paths])
tax_paid_per_year_arr = np.array([p["tax_paid_per_year"] for p in paths])

p50_portfolio = np.percentile(portfolio_arr, 50, axis=0)
p2_5_portfolio = np.percentile(portfolio_arr, 2.5, axis=0)
p97_5_portfolio = np.percentile(portfolio_arr, 97.5, axis=0)
p50_owed = np.percentile(owed_arr, 50, axis=0)
p50_net_gain = np.percentile(net_gain_arr, 50, axis=0)
p2_5_net_gain = np.percentile(net_gain_arr, 2.5, axis=0)
p97_5_net_gain = np.percentile(net_gain_arr, 97.5, axis=0)
p50_tax_yearly = np.percentile(tax_paid_per_year_arr, 50, axis=0)
p2_5_tax_yearly = np.percentile(tax_paid_per_year_arr, 2.5, axis=0)
p97_5_tax_yearly = np.percentile(tax_paid_per_year_arr, 97.5, axis=0)

cumulative_interest = np.cumsum(interest_arr)
cumulative_tax_arr = np.array([p["cumulative_tax"] for p in paths])
p50_cum_tax = np.percentile(cumulative_tax_arr, 50, axis=0)
p2_5_cum_tax = np.percentile(cumulative_tax_arr, 2.5, axis=0)
p97_5_cum_tax = np.percentile(cumulative_tax_arr, 97.5, axis=0)
years = np.arange(SIMULATION_YEARS + 1)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Portfolio size over time
ax = axes[0, 0]
ax.fill_between(years, p2_5_portfolio, p97_5_portfolio, alpha=0.3, label="95% CI (2.5–97.5th %ile)")
ax.plot(years, p50_portfolio, lw=2, label="Median portfolio")
ax.set_ylabel("Portfolio size")
ax.set_xlabel("Year")
ax.set_title(f"Portfolio size over {SIMULATION_YEARS} years")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

# Amount owed over time
ax = axes[0, 1]
ax.plot(years, p50_owed, lw=2, color="C1", label="Loan balance")
ax.set_ylabel("Amount owed")
ax.set_xlabel("Year")
ax.set_title(f"Amount owed (loan balance) over {SIMULATION_YEARS} years")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

# Net worth (amount 'earned' / equity)
ax = axes[1, 0]
ax.fill_between(years, p2_5_net_gain, p97_5_net_gain, alpha=0.3, label="95% CI (2.5–97.5th %ile)")
ax.plot(years, p50_net_gain, lw=2, color="C2", label="Median net gain")
ax.axhline(0, color="gray", ls="--")
ax.set_ylabel("Net worth (portfolio − debt)")
ax.set_xlabel("Year")
ax.set_title(f"Net gain over {SIMULATION_YEARS} years")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

# Combined: portfolio vs owed
ax = axes[1, 1]
ax.plot(years, p50_portfolio, lw=2, label="Median portfolio")
ax.plot(years, p50_owed, lw=2, color="C1", label="Loan balance")
ax.plot(years, p50_net_gain, lw=2, color="C2", label="Net gain")
ax.set_ylabel("Amount")
ax.set_xlabel("Year")
ax.set_title("Portfolio, debt, and net gain")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

plt.tight_layout()
plt.show()


In [ ]:
# Interest and taxes paid over time
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Cumulative interest (same for all paths) and cumulative tax (median + band)
ax = axes[0]
cumulative_interest_plot = np.concatenate([[0], cumulative_interest])
ax.plot(years, cumulative_interest_plot, lw=2, color="C0", label="Cumulative interest paid")
ax.fill_between(years, p2_5_cum_tax, p97_5_cum_tax, alpha=0.3, label="95% CI (2.5–97.5th %ile)")
ax.plot(years, p50_cum_tax, lw=2, color="C1", label="Median cumulative tax")
ax.axhline(0, color="gray", ls="--")
ax.set_ylabel("Amount")
ax.set_xlabel("Year")
ax.set_title(f"Cumulative interest and tax over {SIMULATION_YEARS} years")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

# Yearly interest (fixed) and yearly tax (median + band)
ax = axes[1]
interest_arr_plot = np.concatenate([[0], interest_arr])
ax.bar(years - 0.2, interest_arr_plot, width=0.4, label="Interest paid (yearly)", color="C0", alpha=0.8)
ax.fill_between(years, p2_5_tax_yearly, p97_5_tax_yearly, alpha=0.3, label="95% CI (2.5–97.5th %ile)")
ax.plot(years, p50_tax_yearly, lw=2, color="C1", label="Median tax (yearly)")
ax.axhline(0, color="gray", ls="--")
ax.set_ylabel("Amount")
ax.set_xlabel("Year")
ax.set_title(f"Yearly interest and tax over {SIMULATION_YEARS} years")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.2f}M"))

plt.tight_layout()
plt.show()


In [ ]:
# Summary: assumptions and results
final_net_gain = net_gain_arr[:, -1]
final_portfolio = portfolio_arr[:, -1]
T = SIMULATION_YEARS - 1  # last year index
loan_balance_final = balance_arr[min(T, len(balance_arr)-1)]

print("=" * 60)
print("ASSUMPTIONS")
print("=" * 60)
print(f"  Loan Amount:   {LOAN_AMOUNT:,.0f} at {LOAN_INTEREST_RATE:.2%} over {LOAN_YEARS} years")
print(f"  Loan Price:    {LOAN_PRICE:.2f} (Actual investment: {LOAN_AMOUNT * LOAN_PRICE:,.0f})")
print(f"  Tax rate:      {TAX_RATE:.0%} (lagerprincippet; interest deductible; losses carried forward)")
print(f"  Expenditure:   {YEARLY_EXPENDITURE:,.0f} per year")
print(f"  Simulated:     {SIMULATION_YEARS} years, {N_PATHS} paths")
print(f"  Sampled Years: Starting uniformly from {START_DATE_MIN} onwards")
print()
print("RESULTS — after year", SIMULATION_YEARS)
print("-" * 60)
print(f"  Net gain:     median {np.median(final_net_gain):,.0f}  95% CI [{np.percentile(final_net_gain, 2.5):,.0f}, {np.percentile(final_net_gain, 97.5):,.0f}]")
print(f"  Portfolio:    median {np.median(final_portfolio):,.0f}  95% CI [{np.percentile(final_portfolio, 2.5):,.0f}, {np.percentile(final_portfolio, 97.5):,.0f}]")
print(f"  Loan balance: {loan_balance_final:,.0f}")
print("=" * 60)
